# Regresi Polinomial

Estimasi waktu: **40 menit**

Bagaimana jika hubungan data tidak lurus? Model linear tetap bisa dipakai: tambahkan pangkat dari setiap fitur sebagai fitur baru, lalu latih model linear pada himpunan fitur yang diperluas itu. Tekniknya disebut regresi polinomial.

Ada dua masalah utama pada performa model: *overfitting* dan *underfitting*. Overfitting terjadi saat model terlalu kompleks sehingga bagus di data latih tetapi buruk di data uji. Underfitting terjadi saat model terlalu sederhana sehingga buruk di keduanya.

Overfitting ditangani dengan regularisasi, yang dibahas pada materi berikutnya. Underfitting diatasi dengan model yang lebih kompleks, misalnya regresi polinomial. Bila menambah kompleksitas tetap tidak cukup, datanya yang perlu ditambah atau fiturnya yang perlu diperkaya. Karena itu, langkah awal yang praktis adalah menguji apakah model sanggup meng-overfit data; dari situ kita tahu apakah jumlah data sudah memadai.

## Tujuan

- memahami perbedaan overfitting dan underfitting;
- menerapkan transformasi polinomial pada data;
- melakukan pencarian hyperparameter dengan data validasi.

## Persiapan

Pustaka yang dipakai: `pandas`, `numpy`, `seaborn`, `matplotlib`, dan `scikit-learn`.

In [ ]:
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore")
sns.set_context("notebook")
sns.set_style("white")

Dua fungsi bantu dipakai berulang: satu untuk membandingkan R kuadrat tiap fitur tunggal, satu lagi untuk membandingkan sebaran nilai aktual dan nilai prediksi.

In [ ]:
def r2_per_feature(estimator, features, train, test, target_train, target_test):
    scores_train, scores_test = [], []

    for feature in features:
        estimator.fit(train[[feature]], target_train)
        scores_test.append(estimator.score(test[[feature]], target_test))
        scores_train.append(estimator.score(train[[feature]], target_train))

    positions = np.arange(len(features))
    plt.bar(positions - 0.2, scores_train, width=0.4, label="Train")
    plt.bar(positions + 0.2, scores_test, width=0.4, label="Test")
    plt.xticks(positions, features, rotation=90)
    plt.ylabel("R squared")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Train R2 mean {np.mean(scores_train):.4f} | Test R2 mean {np.mean(scores_test):.4f}")
    print(f"Train R2 max  {np.max(scores_train):.4f} | Test R2 max  {np.max(scores_test):.4f}")

In [ ]:
def plot_actual_vs_fitted(actual, fitted):
    plt.figure()
    sns.kdeplot(x=actual, color="r", label="Actual value")
    sns.kdeplot(x=fitted, color="b", label="Fitted value")
    plt.title("Actual vs Fitted Values")
    plt.xlabel("Price (in dollars)")
    plt.ylabel("Proportion of Cars")
    plt.legend()
    plt.show()

## Membaca data

Data yang dipakai adalah data penjualan mobil yang sudah dibersihkan dan di-encode pada materi regresi linear. Berkas diunduh sekali, lalu dibaca dengan pandas.

In [ ]:
data_path = Path("data/encoded_car_data.csv")
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/encoded_car_data.csv"

if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(data_url, data_path)

cars = pd.read_csv(data_path)
cars.info()

Setelah one-hot encoding, dataset memiliki 35 fitur. Sebelum analisis polinomial, hubungan beberapa fitur dengan target `price` dilihat lebih dulu. Garis orde dua memperlihatkan bentuk yang melengkung, bukan lurus.

In [ ]:
sns.lmplot(x="curbweight", y="price", data=cars, order=2)
plt.show()

sns.lmplot(x="carlength", y="price", data=cars, order=2)
plt.show()

sns.lmplot(x="horsepower", y="price", data=cars, order=2)
plt.show()

## Menyiapkan data

Fitur dan target dipisahkan, lalu data dibagi 70% latih dan 30% uji.

In [ ]:
features = cars.drop(columns="price")
target = cars["price"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, random_state=42
)

print("Jumlah data uji :", X_test.shape[0])
print("Jumlah data latih:", X_train.shape[0])

### Regresi linear dengan semua fitur

Model linear biasa dipakai sebagai pembanding. Selisih R kuadrat latih dan uji menunjukkan seberapa besar overfitting.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

print("R2 data latih:", linear_model.score(X_train, y_train))
print("R2 data uji  :", linear_model.score(X_test, y_test))

In [ ]:
predictions = linear_model.predict(X_test)
plot_actual_vs_fitted(y_test, predictions)

In [ ]:
coefficients = dict(zip(features.columns, linear_model.coef_))
coefficients

Dua koefisien pertama nilainya sangat besar sehingga tidak muat di plot. Koefisien sisanya digambarkan terpisah.

In [ ]:
plt.bar(features.columns[2:], np.abs(linear_model.coef_[2:]))
plt.xticks(rotation=90)
plt.ylabel("coefficients")
plt.tight_layout()
plt.show()

Koefisien besar sering dianggap lebih penting, tetapi tidak selalu begitu. Karena itu, tiap fitur diuji satu per satu.

In [ ]:
r2_per_feature(linear_model, list(features.columns), X_train, X_test, y_train, y_test)

Beberapa fitur tunggal memberi R kuadrat yang cukup dekat dengan penggunaan seluruh fitur. Koefisien yang membengkak tidak menjamin prediksi lebih baik, sehingga pentingnya fitur tidak bisa dibaca hanya dari besar koefisien.

### Latihan: pipeline dengan standardisasi

Hitung R kuadrat memakai `Pipeline` berisi `StandardScaler` dan `LinearRegression`, lalu bandingkan sebaran prediksi dengan nilai aktual.

In [ ]:
scaled_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

scaled_pipeline.fit(X_train, y_train)
print("R2 data latih:", scaled_pipeline.score(X_train, y_train))
print("R2 data uji  :", scaled_pipeline.score(X_test, y_test))

plot_actual_vs_fitted(y_test, scaled_pipeline.predict(X_test))

Pipeline yang sama juga bisa dipakai untuk menghitung R kuadrat per fitur.

In [ ]:
r2_per_feature(scaled_pipeline, list(features.columns), X_train, X_test, y_train, y_test)

## Fitur polinomial

Transformasi polinomial menambah kompleksitas model, tetapi risikonya overfitting ikut naik. Berikut transformasi derajat dua: fitur asli, kuadratnya, dan semua suku silang antarfitur.

In [ ]:
polynomial = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = polynomial.fit_transform(X_train)
X_test_poly = polynomial.transform(X_test)

print("Bentuk data latih:", X_train_poly.shape)
print("Bentuk data uji  :", X_test_poly.shape)

In [ ]:
poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)

print("R2 data latih:", poly_model.score(X_train_poly, y_train))
print("R2 data uji  :", poly_model.score(X_test_poly, y_test))

R kuadrat data latih naik ke sekitar 0,98, sedangkan data uji tertahan di sekitar 0,84. Selisih selebar itu adalah tanda overfitting: model menghafal data latih dan tidak ikut membaik pada data baru.

### Pipeline polinomial

Langkah yang sama bisa dirangkai dalam `Pipeline` agar lebih ringkas.

In [ ]:
poly_pipeline = Pipeline(steps=[
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
    ("model", LinearRegression()),
])

poly_pipeline.fit(X_train, y_train)
print("R2 data latih:", poly_pipeline.score(X_train, y_train))
print("R2 data uji  :", poly_pipeline.score(X_test, y_test))

In [ ]:
r2_per_feature(poly_pipeline, list(features.columns), X_train, X_test, y_train, y_test)

R kuadrat tertinggi dari fitur tunggal masih di bawah R kuadrat saat seluruh fitur dipakai bersama. Transformasi polinomial pada satu fitur saja belum menangkap informasi gabungan antarfitur.

## GridSearch dengan pipeline

`GridSearchCV` mencari kombinasi hyperparameter terbaik lewat validasi silang. Nama parameter pipeline ditulis dengan pola `namalangkah__parameter`. Secara bawaan dipakai 5-fold cross validation.

In [ ]:
search_pipeline = Pipeline(steps=[
    ("polynomial", PolynomialFeatures(include_bias=False, degree=2)),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

param_grid = {"polynomial__degree": [1, 2, 3]}

search = GridSearchCV(search_pipeline, param_grid, n_jobs=1)
search.fit(X_train, y_train)

best_model = search.best_estimator_
best_model

In [ ]:
print("R2 data uji:", best_model.score(X_test, y_test))
plot_actual_vs_fitted(y_test, best_model.predict(X_test))

GridSearch memilih derajat 1, sehingga R kuadrat data uji sama dengan model linear penuh, sekitar 0,84. Derajat yang lebih tinggi hanya menaikkan skor data latih tanpa memperbaiki data uji, gejala overfitting.

### Latihan: GridSearch per fitur

Ulangi pencarian hyperparameter untuk setiap fitur tunggal, lalu bandingkan R kuadrat latih dan uji.

In [ ]:
def grid_r2_per_feature(features, train, test, target_train, target_test):
    scores_train, scores_test = [], []

    for feature in features:
        grid_pipeline = Pipeline(steps=[
            ("polynomial", PolynomialFeatures(include_bias=False, degree=2)),
            ("model", LinearRegression()),
        ])
        grid = GridSearchCV(
            grid_pipeline,
            {"polynomial__degree": [1, 2, 3, 4, 5]},
            n_jobs=1,
        )
        grid.fit(train[[feature]], target_train)
        best = grid.best_estimator_

        scores_train.append(best.score(train[[feature]], target_train))
        scores_test.append(best.score(test[[feature]], target_test))

    positions = np.arange(len(features))
    plt.bar(positions - 0.2, scores_train, width=0.4, label="Train")
    plt.bar(positions + 0.2, scores_test, width=0.4, label="Test")
    plt.xticks(positions, features, rotation=90)
    plt.ylabel("R squared")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Train R2 mean {np.mean(scores_train):.4f} | Test R2 mean {np.mean(scores_test):.4f}")
    print(f"Train R2 max  {np.max(scores_train):.4f} | Test R2 max  {np.max(scores_test):.4f}")


grid_r2_per_feature(list(features.columns), X_train, X_test, y_train, y_test)

## Ringkasan

Menambah pangkat fitur membuat model linear mampu menangkap pola melengkung. Namun derajat yang terlalu tinggi membuat model menghafal data latih, sehingga R kuadrat data uji jatuh. Pemilihan derajat lewat validasi silang menyeimbangkan keduanya.